# Bike Sharing Demand: Statistical Analysis
**Author:** Min Khaung Min Lwin

**Dataset:** Bike Sharing Dataset (day.csv), UCI Machine Learning Repository — https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset


## 1. Data Loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

df = pd.read_csv("bike+sharing+dataset/day.csv")
df.head()

,instant,dteday,season,yr,mnth,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,6,0,2,0.344167,0.363625,0.805833,0.160446,331,654,985
1,2,2011-01-02,1,0,1,0,0,0,2,0.363478,0.353739,0.696087,0.248539,131,670,801
2,3,2011-01-03,1,0,1,0,1,1,1,0.196364,0.189405,0.437273,0.248309,120,1229,1349
3,4,2011-01-04,1,0,1,0,2,1,1,0.200000,0.212122,0.590435,0.160296,108,1454,1562
4,5,2011-01-05,1,0,1,0,3,1,1,0.226957,0.229270,0.436957,0.186900,82,1518,1600


In [2]:
print(df.shape)
print(df.dtypes)
print(df.isna().sum())
df.describe().T

(731, 16)
instant         int64
dteday            str
season          int64
yr              int64
mnth            int64
holiday         int64
weekday         int64
workingday      int64
weathersit      int64
temp          float64
atemp         float64
hum           float64
windspeed     float64
casual          int64
registered      int64
cnt             int64
dtype: object
instant       0
dteday        0
season        0
yr            0
mnth          0
holiday       0
weekday       0
workingday    0
weathersit    0
temp          0
atemp         0
hum           0
windspeed     0
casual        0
registered    0
cnt           0
dtype: int64


,count,mean,std,min,25%,50%,75%,max
instant,731.0,366.000000,211.165812,1.000000,183.500000,366.000000,548.500000,731.000000
season,731.0,2.496580,1.110807,1.000000,2.000000,3.000000,3.000000,4.000000
yr,731.0,0.500684,0.500342,0.000000,0.000000,1.000000,1.000000,1.000000
mnth,731.0,6.519836,3.451913,1.000000,4.000000,7.000000,10.000000,12.000000
holiday,731.0,0.028728,0.167155,0.000000,0.000000,0.000000,0.000000,1.000000
weekday,731.0,2.997264,2.004787,0.000000,1.000000,3.000000,5.000000,6.000000
workingday,731.0,0.683995,0.465233,0.000000,0.000000,1.000000,1.000000,1.000000
weathersit,731.0,1.395349,0.544894,1.000000,1.000000,1.000000,2.000000,3.000000
temp,731.0,0.495385,0.183051,0.059130,0.337083,0.498333,0.655417,0.861667
atemp,731.0,0.474354,0.162961,0.079070,0.337842,0.486733,0.608602,0.840896


In [3]:
# Categorical variables are stored as integer codes; check which values appear
for col in ["season", "yr", "mnth", "holiday", "weekday", "workingday", "weathersit"]:
    print(col, df[col].value_counts().sort_index().to_dict())

season {1: 181, 2: 184, 3: 188, 4: 178}
yr {0: 365, 1: 366}
mnth {1: 62, 2: 57, 3: 62, 4: 60, 5: 62, 6: 60, 7: 62, 8: 62, 9: 60, 10: 62, 11: 60, 12: 62}
holiday {0: 710, 1: 21}
weekday {0: 105, 1: 105, 2: 104, 3: 104, 4: 104, 5: 104, 6: 105}
workingday {0: 231, 1: 500}
weathersit {1: 463, 2: 247, 3: 21}


------

This project looks at 731 days of Capital Bikeshare rentals in Washington, DC, from 2011 and 2012. I want to find out whether the weather on a given day affects how many bikes are rented. Knowing how much demand drops on bad-weather days would help an operator plan how many bikes and staff to have available.

### Research question and plan
**Question**: Does the number of bikes rented per day (cnt) differ between clear, misty/cloudy and light rain/snow days?

**Hypotheses**:

- `H₀`: mean daily rentals are the same in all three weather groups.
- `H₁`: at least one group's mean differs.
**Plan**: A one-way ANOVA compares the means of three or more independent groups, so it's my first choice. It assumes each group is roughly normal and the groups have similar variances. I'll check these with the Shapiro–Wilk and Levene tests. If either assumption fails, I'll use the Kruskal–Wallis test instead, which doesn't need normality. This is especially relevant because the light rain/snow group has only 21 days.

**Limitations to keep in mind**:

- Consecutive days aren't independent, and ridership grew a lot from 2011 to 2012.
- Weather overlaps with year and season. Most bad-weather days fall in 2011 and in autumn, so part of any difference may come from those rather than from the weather itself.


## 2. Data Preparation

In [4]:
# Check the Readme's season codes against the calendar before using them as labels
display(pd.crosstab(df["mnth"], df["season"]))

# Find the exact dates where the season code changes
change = df["season"].ne(df["season"].shift())
df.loc[change, ["dteday", "season"]]

season,1,2,3,4
mnth,,,,
1,62,0,0,0
2,57,0,0,0
3,40,22,0,0
4,0,60,0,0
5,0,62,0,0
6,0,40,20,0
7,0,0,62,0
8,0,0,62,0
9,0,0,44,16


,dteday,season
0,2011-01-01,1
79,2011-03-21,2
171,2011-06-21,3
265,2011-09-23,4
354,2011-12-21,1
445,2012-03-21,2
537,2012-06-21,3
631,2012-09-23,4
720,2012-12-21,1


### Note: the Readme's season labels don't match the data

Readme.txt maps the season codes as `1: spring, 2: summer, 3: fall, 4: winter`. Under that mapping, 2011-01-27 would be labelled "Spring" and 2012-10-29 "Winter", which can't be right.

The checks above show what the codes actually follow:

- Code 1 covers all of January and February, most of March and the last part of December.
- Code 4 covers late September, all of October and November, and most of December.
- The code changes on the same dates both years: 21 March, 21 June, 23 September and 21 December. Those are the equinoxes and solstices.

So the codes follow **astronomical seasons**, and code 1 is winter, not spring. The Readme labels are shifted by one season. I use the mapping that matches the data: `1: Winter, 2: Spring, 3: Summer, 4: Fall`.

This changes only the labels, not the grouping: each code still covers the same days. It does change how any seasonal result is read, though. The lowest-demand season is winter, not spring.

Checking the metadata against the data before relying on it is the kind of data screening Lusa et al. (2024) recommend.

In [5]:
# Convert the date and map integer codes to readable labels
# (season labels corrected from Readme.txt; see the note above)
df["dteday"] = pd.to_datetime(df["dteday"])
df["season_label"] = df["season"].map({1: "Winter", 2: "Spring", 3: "Summer", 4: "Fall"})
df["weather_label"] = df["weathersit"].map({1: "Clear", 2: "Mist/Cloudy", 3: "Light rain/snow"})
df["year"] = df["yr"].map({0: 2011, 1: 2012})

# Inspect the suspicious values found in describe()
display(df.loc[df["hum"] == 0, ["dteday", "weather_label", "hum", "cnt"]])
display(df.nsmallest(5, "cnt")[["dteday", "weather_label", "season_label", "cnt"]])

,dteday,weather_label,hum,cnt
68,2011-03-10,Light rain/snow,0.0,623


,dteday,weather_label,season_label,cnt
667,2012-10-29,Light rain/snow,Fall,22
26,2011-01-27,Clear,Winter,431
725,2012-12-26,Light rain/snow,Winter,441
25,2011-01-26,Light rain/snow,Winter,506
64,2011-03-06,Mist/Cloudy,Winter,605


### Screening the two suspicious values

**`hum = 0.0` on 2011-03-10: recording error, set to missing.** Zero relative humidity isn't physically possible outdoors, and the same day is coded as light rain/snow, which makes it even less plausible. The days either side have humidity of 0.78 and 0.65. I treat it as a recording error and set it to `NaN`. The test below doesn't use humidity, so this doesn't change any results.

**`cnt = 22` on 2012-10-29: real extreme-weather day, kept.** This is the day Hurricane Sandy reached the US East Coast (Blake et al., 2013). Rentals fell from 4,459 the day before to 22 and only partly recovered the next day (1,096). The near-shutdown matches a real event, so this isn't an error. Removing it would artificially shrink the weather effect, so I keep it.

The dataset codes a hurricane as just "light rain/snow". That shows how coarse the weather coding is: one category covers everything from drizzle to a hurricane. I return to this under limitations.

> Blake, E. S., Kimberlain, T. B., Berg, R. J., Cangialosi, J. P., & Beven, J. L. (2013). *Tropical cyclone report: Hurricane Sandy (AL182012), 22–29 October 2012*. National Hurricane Center.

In [6]:
# Zero humidity is a recording error: set it to missing
df.loc[df["hum"] == 0, "hum"] = np.nan
print("Missing humidity values:", df["hum"].isna().sum())

Missing humidity values: 1


## 3. Descriptive Statistics

In [7]:
def describe_by_group(df, group_col, value_col="cnt"):
    """Return count, mean, median, std, min and max of value_col for each group."""
    return (df.groupby(group_col)[value_col]
              .agg(["count", "mean", "median", "std", "min", "max"])
              .round(1))

display(describe_by_group(df, "weather_label"))
display(describe_by_group(df, "season_label"))
display(describe_by_group(df, "year"))

# Is bad weather spread evenly across seasons?
display(pd.crosstab(df["season_label"], df["weather_label"]))

# Is bad weather spread evenly across the two years? (year is a confounder)
display(pd.crosstab(df["year"], df["weather_label"]))

,count,mean,median,std,min,max
weather_label,,,,,,
Clear,463,4876.8,4844.0,1879.5,431,8714
Light rain/snow,21,1803.3,1817.0,1240.3,22,4639
Mist/Cloudy,247,4035.9,4040.0,1809.1,605,8362


,count,mean,median,std,min,max
season_label,,,,,,
Fall,178,4728.2,4634.5,1699.6,22,8555
Spring,184,4992.3,4941.5,1696.0,795,8362
Summer,188,5644.3,5353.5,1459.8,1115,8714
Winter,181,2604.1,2209.0,1399.9,431,7836


,count,mean,median,std,min,max
year,,,,,,
2011,365,3405.8,3740.0,1378.8,431,6043
2012,366,5599.9,5927.0,1788.7,22,8714


weather_label,Clear,Light rain/snow,Mist/Cloudy
season_label,,,
Fall,103,10,65
Spring,113,3,68
Summer,136,4,48
Winter,111,4,66


weather_label,Clear,Light rain/snow,Mist/Cloudy
year,,,
2011,226,15,124
2012,237,6,123


### What the descriptive statistics show

| Finding | Evidence | What it means |
|---|---|---|
| Strong weather gradient | Mean rentals per day: Clear 4,877 → Mist/Cloudy 4,036 → Light rain/snow 1,803 | Light rain/snow days average about 63% fewer rentals than clear days |
| Groups are roughly symmetric | Within each weather group, mean ≈ median (e.g., 1,803 vs 1,817) | Normality may be reasonable, but it still needs testing per group |
| Spreads are similar | SDs range from 1,240 to 1,880, so the largest is about 1.5 times the smallest | Equal variances are plausible; confirm with Levene's test |
| Large growth between years | 2011 mean 3,406 vs 2012 mean 5,600, a 64% increase | Year is a major confounder |
| Bad weather is concentrated in 2011 | 15 of the 21 light rain/snow days are in 2011, the lower-demand year; Clear and Mist/Cloudy days are split almost evenly | Part of the light rain/snow gap is a year effect, so the raw comparison overstates the weather effect |
| Bad weather clusters in one season | 10 of the 21 light rain/snow days are in Fall (season code 4) | Part of the weather effect may be a season effect |